# Additional Cleaning and Merging
## Data

- **brooklyn_selected_zips.csv**  
  Cleaned census data for the relevant ZIP codes - this is mostly ready to go, and won't be used until step 4

- **311_2012-2022_cleaned.csv**  
  Semi-cleaned 311 data for the relevant ZIP codes  

## Steps

0. Load packages and import data  
1. Add necessary columns (duration, year, zipyear) and remove any other unnecessary columns.
2. Prepare the columns to be counted by categorizing and sorting
3. Create subject-specific dataframes containing categorical counts by year and zip
4. Prepare census data, combine the 311 dataframes created in step 4, and merge and save the data.

## 0) import packages and data

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

data311 = pd.read_csv("raw311data.csv")
census = pd.read_csv("brooklyn_selected_zips.csv")

/var/folders/b2/bbg1y6wx7z15pdkgv123vdhr0000gn/T/ipykernel_39922/2744050735.py:6: DtypeWarning: Columns (0: Incident Zip) have mixed types. Specify dtype option on import or set low_memory=False.
  data311 = pd.read_csv("raw311data.csv")


## 1) Pare down the 311 data even futher
In order to eventually merge, we'll need to consolidate to one row per zip code per year, which means converting the current data to counts rather than raw entries. 

First, though, we need to get rid of any columns that we don't plan to do so for. We also need to create the YEAR and ZIPYEAR columns, as well as the duration variable calculated using Created Date and Closed Date as time points.

### 1a) Clean up the columns
Remove columns that won't coerce in a useful way when we run counts, and rename the rest without white space or special characters.

In [2]:
pared311old = data311.drop(columns=['Unique Key', 'Problem Detail (formerly Descriptor)',
                                   'Additional Details','Location Type', 'Incident Address',
                                   'Resolution Description', 'BBL', 'Open Data Channel Type',
                                   'Location', 'Latitude', 'Longitude'])

# PRINT CALL
# print(pared311old.columns)

pared311old = pared311old.rename(columns={"Problem (formerly Complaint Type)":"problem"})
pared311old = pared311old.rename(columns={"Agency Name":"agency"})
pared311old = pared311old.rename(columns={"Council District":"district"})
pared311old = pared311old.rename(columns={"Police Precinct":"precinct"})

# PRINT CALL
# print(pared311old.columns)

### 1b) Extract the year from the Created Date column
Create the year variable for grouping and for use in the ZIPYEAR column, which we will use to join datasets later.

In [3]:
## MERGE PREP

## rename zip code columns
pared311old = pared311old.rename(columns={"Incident Zip":"ZIP"})

## check that it worked
print(pared311old.columns)


## create a year column for the 311 dataset (from incident date)
pared311old['Created Date'] = pd.to_datetime(
    pared311old['Created Date'],
    errors='coerce'
)

pared311old['YEAR'] = pared311old['Created Date'].dt.year



# sort by YEAR in ascending order
pared311old = pared311old.sort_values(by='YEAR').reset_index(drop=True)
print(pared311old['YEAR'].unique())


/var/folders/b2/bbg1y6wx7z15pdkgv123vdhr0000gn/T/ipykernel_39922/820676159.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pared311old['Created Date'] = pd.to_datetime(


Index(['Created Date', 'Closed Date', 'agency', 'problem', 'ZIP', 'Status',
       'district', 'precinct'],
      dtype='str')
[2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022]


### 1c) Create the ZIPYEAR variable
Combine the YEAR and ZIP columns into a single variable.

In [4]:
# coerce any 9-digit ZIPs down to their 5-digit base first
pared311old['ZIP'] = pared311old['ZIP'].astype(str).str[:5]

# create the ZIPYEAR column by converting both to strings and adding a space
pared311old['ZIPYEAR'] = pared311old['ZIP'].astype(str) + ' ' + pared311old['YEAR'].astype(str)

### 1b) Create the duration of complaint variable
Calculate the duration for which a given complaint was unresolved from when the complaint was made (Created Date) to when it was marked as resolved (Closed Date).

In [5]:
pared311 = pared311old

## ensure columns are in datetime format
pared311['Created Date'] = pd.to_datetime(pared311['Created Date'])
pared311['Closed Date'] = pd.to_datetime(pared311['Closed Date'])

## calculate duration in minutes
pared311['duration'] = (pared311['Closed Date'] - pared311['Created Date']).dt.total_seconds() / 60

## filter: remove negative durations (impossible data)
pared311 = pared311[pared311['duration'] >= 0]

## filter: remove outliers above the 99th percentile (the "zombie tickets")
# Calculate the 99th percentile for each year individually
pared311 = pared311.groupby('YEAR', group_keys=False).apply(
    lambda x: x[x['duration'] <= x['duration'].quantile(0.99)]
)

## check the statistics 
# print(pared311['duration'].describe().apply(lambda x: format(x, 'f')))

## now we can drop the original date columns
pared311 = pared311.drop(columns=['Created Date', 'Closed Date', 'Status'])

## check again
# print(minimal311.head())

pared311.to_csv('preCategories.csv', index=False)


## 2) Prepping the columns to be counted
First, starting into a copied dataset for posterity.

In [6]:
minimal311 = pared311

### 2a) Cleaning the precinct column
The precinct col is already pretty well organized, so we don't need to run a unique here. Instead, we just want to strip "Precinct " from each entry. After that, we can convert to numeric and be done!

In [7]:
import re

def final_cleanup(df):

    # strip "Precinct " from the entries (with format "Precinct ##")
    df['precinct'] = df['precinct'].astype(str).str.replace(r'(?i)Precinct\s*', '', regex=True)
    
    # trim any remaining whitespace 
    df['precinct'] = df['precinct'].str.strip()
    
    # check for non-numeric leftovers (e.g., "N/A", "Unknown", or empty)
    is_numeric = df['precinct'].str.match(r'^\d+$', na=False)
    error_count = len(df) - is_numeric.sum()
    
    if error_count == 0:
        df['precinct'] = pd.to_numeric(df['precinct'])
        print("yay all rows converted to numeric")
    else:
        print(f"still found {error_count} problematic cases.")
        print("non numeric entries:")
        print(df.loc[~is_numeric, 'precinct'].unique()[:10])
        
    return df


# now run it
final_cleanup(minimal311)

## uh oh, still have lots that aren't numbers...

## filter for rows that aren't digits and look at the unique values again
mismatches = minimal311[~minimal311['precinct'].astype(str).str.match(r'^\d+$', na=False)]
# print(mismatches['precinct'].value_counts())

## okay, since all of the remaining entries are the same and just say "Unspecified" we can handle
## all of them at once by recoding to 0 (which will clearly distinguish them from the other entries)

# replace the "Unspecified" strings with '0'
minimal311['precinct'] = minimal311['precinct'].replace('Unspecified', '0')

# convert the entire column to numeric again
minimal311['precinct'] = pd.to_numeric(minimal311['precinct']).astype(int)

# check it again my printing
# print(f"column 'precinct' is now: {minimal311['precinct'].dtype}")
# print(f"count of zeros: {(minimal311['precinct'] == 0).sum()}")

still found 7532 problematic cases.
non numeric entries:
<StringArray>
['Unspecified']
Length: 1, dtype: str


### 2b) Preparing the agency column for counts.

In [8]:
# okay first let's see what we're working with using unique to print all the unique values in agency
pd.set_option('display.max_rows', None)
unique_agencies = minimal311['agency'].unique()
# print(unique_agencies)

Unfortunately there are over a hundred unique values, which is way too many to be reformatted as we intend to. However, we can't just get rid of this column, because agency may be a relevant factor later. Instead, based on the official NYC Government Organizational Chart, we grouped the agencies into categories below. Luckily, many of the unique entries were individual schools or specific offices within agencies, so we didn't have to do too much qualitative grouping, and when we did we were able to reference academically sound groupings.

#### 2bi) creating a dictionary to group the unique entries


In [9]:
agency_map = {
    
    # DEPARTMENT OF EDUCATION 
    'Department of Education': 'doe',
    # Note: The code below uses a loop to catch all "School - " entries
    
    # PUBLIC SAFETY
    'New York City Police Department': 'safety',
    'Taxi and Limousine Commission': 'safety',
    "Mayor's Office of Special Enforcement": 'safety',
    
    # INFRASTRUCTURE & ENVIRONMENT 
    'Department of Sanitation': 'infr_env',
    'Department of Buildings': 'infr_env',
    'Department of Environmental Protection': 'infr_env',
    'Department of Transportation': 'infr_env',
    'Department of Parks and Recreation': 'infr_env',
    'Traffic Management Center': 'infr_env',
    
    # SOCIAL SERVICES, HEALTH, HOUSING 
    'Department of Housing Preservation and Development': 'ss_hous',
    'Department of Health and Mental Hygiene': 'ss_hous',
    'Department of Homeless Services': 'ss_hous',
    'Operations Unit - Department of Homeless Services': 'ss_hous',
    'DHS Advantage Programs': 'ss_hous',
    'Department for the Aging': 'ss_hous',
    'HealthCare Connections': 'ss_hous',
    
    # FINANCE, TAX & PROPERTY (The 'Internal Units') 
    'Land Records': 'fin_prop',
    'Commercial Exemption Unit': 'fin_prop',
    'Correspondence Unit': 'fin_prop',
    'Senior Citizen Rent Increase Exemption Unit': 'fin_prop',
    'Property Exec Office': 'fin_prop',
    'Personal Exemption Unit': 'fin_prop',
    'RPIE Unit': 'fin_prop',
    'Office of the Taxpayer Advocate': 'fin_prop',
    'Payment Operations': 'fin_prop',
    'Condo or CoOp Unit': 'fin_prop',
    'Valuation Policy': 'fin_prop',
    'Disability Rent Increase Exemption Unit': 'fin_prop',
    'Refunds and Adjustments': 'fin_prop',
    'Discrepancy and Billing': 'fin_prop',
    'DRIE Unit': 'fin_prop',
    'DOF Legal Affairs - Taxpayer Services Unit': 'fin_prop',
    'Exemption Unit': 'fin_prop',
    'Accounts Examination': 'fin_prop',
    'Exemption Policy Unit': 'fin_prop',
    
    # BUSINESS & TECH 
    'Economic Development Corporation': 'biz_tech',
    'Department of Consumer Affairs': 'biz_tech',
    'Department of Consumer and Worker Protection': 'biz_tech',
    'Department of Information Technology and Telecommunications': 'biz_tech',
    'Office of Technology and Innovation': 'biz_tech',
    '3-1-1 Call Center': 'biz_tech',
    'External Affairs Unit': 'biz_tech'
}

#### 2bii) creating and applying a function to add a column which groups the problems as specified above

In [10]:
# now that we've established the dictionary, we can make a function to use it

def clean_agency(name):

    # catch-all for all individual schools
    if str(name).startswith('School -'):
        return 'school'
    
    # look up in the map, default to 'Other' if not found
    return agency_map.get(name, 'Other/General')

# run the function
minimal311['agency_group'] = minimal311['agency'].apply(clean_agency)

# check what that leaves us with
# print(minimal311['agency_group'].value_counts())

### 2c) the big one: the problem column

In [11]:
# oh boy okay let's see how many unique problems we have...
pd.set_option('display.max_rows', None)
unique_probs = minimal311['problem'].unique()
# print(unique_probs)

#### 2ci) creating a function to sort the unique problems based on the words they contain
There are 261 unique entries in this column, and we don't have time to sort them all on a case-by-case basis. Instead, we'll sort into industry standard groups based on keywords. 

In [12]:
def categorize_problem(problem):
    p = str(problem).lower()
    
    # HOUSING & BUILDINGS
    if any(x in p for x in ['heat', 'water leak', 'plumbing', 'paint', 'elevator', 'boiler', 'mold', 'unsanitary', 'appliance', 'housing']):
        return 'housing_building'
    
    # NOISE & QUALITY OF LIFE
    elif 'noise' in p or 'drinking' in p or 'urinating' in p or 'smoke' in p:
        return 'noise_qol'
    
    # SANITATION & ENVIRONMENTAL
    elif any(x in p for x in ['collection', 'sanitation', 'litter', 'recycling', 'dirty', 'sweeping', 'sewage', 'asbestos', 'air quality']):
        return 'sanitation_env'
    
    # STREETS, SIDEWALKS, & TRAFFIC
    elif any(x in p for x in ['street', 'sidewalk', 'traffic', 'parking', 'curb', 'highway', 'bridge', 'pothole']):
        return 'street_traffic'
    
    # PUBLIC SAFETY & POLICING
    elif any(x in p for x in ['abandoned vehicle', 'derelict', 'drug', 'emergency', 'police', 'fireworks', 'illegal', 'homeless']):
        return 'public_safety'
    
    # PARKS & TREES
    elif any(x in p for x in ['tree', 'park', 'animal', 'pigeon', 'bee', 'wasp', 'plant']):
        return 'parks_nature'
    
    # FINANCE & PROPERTY
    elif any(x in p for x in ['dof', 'scrie', 'drie', 'taxpayer', 'advocate', 'exemption', 'legal']):
        return 'finance_legal'
    
    # BUSINESS & CONSUMER AFFAIRS
    elif any(x in p for x in ['consumer', 'vending', 'taxi', 'for hire', 'food', 'day care', 'pet shop']):
        return 'business_consumer'
    
    # TECH & INFRASTRUCTURE
    elif any(x in p for x in ['tech', 'telecomm', 'linknyc', 'water system', 'electric', 'gas']):
        return 'tech_infrastructure'
    
    # SCHOOLS
    elif 'school' in p:
        return 'education'
    
    return 'other_misc'


# lets run it!
minimal311['problem_group'] = minimal311['problem'].apply(categorize_problem)
# print(minimal311['problem_group'].unique())

# and just to be safe, let's check what's in other_misc:
other_list = sorted(minimal311[minimal311['problem_group'] == 'other_misc']['problem'].unique().tolist())
# print(other_list)

# okay well now that i pandora's boxed it i have to deal with it

#### 2cii) further sorting the other_misc category

In [13]:
# gonna sort as best i can may still need an other_misc category though

# social disorder
social_keywords = 'homeless|encampment|drug|graffiti|panhandling|urinating|disorderly|squeegee|vending'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(social_keywords, case=False, na=False)),
    'problem_group'
] = 'social_disorder'

# specific infrastructure/property issues
infrastructure_keywords = 'bike rack|muni meter|bus stop|toilet|payphone|street light|sign'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(infrastructure_keywords, case=False, na=False)),
    'problem_group'
] = 'public_space_amenity'

# any missed housing/building complaints
building_keywords = 'door/window|flooring|stairs|construction|home repair'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(building_keywords, case=False, na=False)),
    'problem_group'
] = 'housing_building'

# any missed health & safety complaints
health_keywords = 'covid|vaccine|mask|hazardous|radioactive|asbestos|lead'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(health_keywords, case=False, na=False)),
    'problem_group'
] = 'health_safety'


# ok now lets see how many are still left in the 'other' bucket
# print(minimal311[minimal311['problem_group'] == 'other_misc']['problem'].value_counts())

#### 2ciii) final run through other_misc to sort the most common remaining complaints

In [14]:

# housing & buildings (NONCONST, Building/Use, SAFETY, SPIT, etc.)
housing_refine = 'NONCONST|Building/Use|Maintenance|SAFETY|DOOR/WINDOW|FLOORING|OUTSIDE BUILDING|Construction|Scaffold|SPIT|Marshal|Home Repair'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(housing_refine, case=False, na=False)),
    'problem_group'
] = 'housing_building'

# public space complaints (Blocked Driveway, Muni Meters, Bus Stops, etc.)
public_space_refine = 'Blocked Driveway|Muni Meter|Parking|Bus Stop|Toilet|Payphone|Bike|Ferry|Outdoor Dining'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(public_space_refine, case=False, na=False)),
    'problem_group'
] = 'public_space'

# environmental & sanitation (Rodent, Sewer, Water Conservation, Snow, etc.)
env_refine = 'Rodent|Sewer|Waste|Snow|Recycling|Dumping|Water Quality|Water Conservation|Standing Water|Vacant Lot|Mosquito'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(env_refine, case=False, na=False)),
    'problem_group'
] = 'sanitation_env'

# public safety & health enforcement (NonCompliance, Fireworks, Drug, etc.)
enforce_refine = 'Enforcement|NonCompliance|Disorderly|Drug|Panhandling|Urinating|Squeegee|Vending|Graffiti|Homeless|Encampment|Safety'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(enforce_refine, case=False, na=False)),
    'problem_group'
] = 'public_safety'

# admin & finance (Literature Request, DOF, Taxpayer, etc.)
admin_refine = 'Literature|Inquiry|DOF|Taxpayer|Advocate|Account|Refund|Data|IAD|Borough Office|Property Value'
minimal311.loc[
    (minimal311['problem_group'] == 'other_misc') & 
    (minimal311['problem'].str.contains(admin_refine, case=False, na=False)),
    'problem_group'
] = 'finance_legal'


# CHECK ONE LAST TIME
# print("TOP 5 LEFT IN OTHER_MISC")
# print(minimal311[minimal311['problem_group'] == 'other_misc']['problem'].value_counts().head(5))

# print("\n--- FINAL GROUP DISTRIBUTION ---")
# print(minimal311['problem_group'].value_counts())

#### 2civ) assigning the groups made from other_misc to the original categories

In [15]:
# final step! regrouping the groups we made from the other_misc top problems into the original groups. say groups one more time.


# create a mapping dictionary to fold sub-groups into master groups

consolidation_map = {
    # for public_safety
    'social_disorder': 'public_safety',
    
    # for finance_legal
    'admin_finance': 'finance_legal',
    
    # for housing_building
    'health_safety': 'housing_building',
    
    # for street_traffic (usually where physical assets are grouped)
    'public_space_amenity': 'street_traffic',
    
    # for  public_space 
    'public_space': 'street_traffic'
}

# apply the mapping (BUT! only to the groups that need to change)
minimal311['problem_group'] = minimal311['problem_group'].replace(consolidation_map)

# one last cleanup for the last few major 'other_misc' items 
minimal311.loc[minimal311['problem'] == 'DPR Internal', 'problem_group'] = 'parks_nature'
minimal311.loc[minimal311['problem'] == 'Obstruction', 'problem_group'] = 'street_traffic'
minimal311.loc[minimal311['problem'] == 'Residential Disposal Complaint', 'problem_group'] = 'sanitation_env'

# FINAL CHECK:
# print(minimal311['problem_group'].value_counts())

## 3) Creating the annual dataset with counts and proportions

Now that we have reasonably grouped the variables, we can start converting this huge dataset into a much smaller one with counts instead of individual entries. We will create a new datset using raw counts. Note to edit out earlier mentions of percent because honestly we can figure that out later and it's 1am so just creating the dataset for merging purposes will get the job done. To keep things modular in case of errors, we'll create the data frames for each category, then append them to each other later.

### 3a) agency count data frame

In [16]:
# print(minimal311['agency_group'].unique())
# a reminder of our categories: safety, infr_env, ss_hous, biz_tech, doe, fin_prop, school

# create the contingency table (cross-tabulation)
AgencyFrame = pd.crosstab(minimal311['ZIPYEAR'], minimal311['agency_group'])

# let's alphabetize bc why not
desired_cols = ['biz_tech', 'doe', 'fin_prop', 'infr_env', 'safety', 'school', 'ss_hous']
AgencyFrame = AgencyFrame.reindex(columns=desired_cols, fill_value=0)

# check how it did
AgencyFrame.head()


agency_group,biz_tech,doe,fin_prop,infr_env,safety,school,ss_hous
ZIPYEAR,,,,,,,
11212 2012,252,0,290,4124,3311,14,11189
11212 2013,194,0,201,4009,3336,24,10631
11212 2014,189,0,244,4417,3523,13,12511
11212 2015,172,0,277,4827,4316,8,11609
11212 2016,128,0,212,4858,5281,22,11601


### 3b) duration AVERAGES data frame

In [17]:
durationFixed = pared311old

# calculate duration in seconds first for higher precision
durationFixed['duration_sec'] = (durationFixed['Closed Date'] - durationFixed['Created Date']).dt.total_seconds()

# filter: keep only tickets that took at least 1 second to close 
## (this removes the "instant" 0.0 entries which were bugging the system)
refined_duration = durationFixed[durationFixed['duration_sec'] > 0].copy()

## convert to minutes
refined_duration['duration_min'] = refined_duration['duration_sec'] / 60


## group by ZIPYEAR for the average
DurationFrame = refined_duration.groupby('ZIPYEAR')['duration_min'].mean().reset_index()
DurationFrame = DurationFrame.rename(columns={'duration_min': 'avg_duration_min'})
DurationFrame.set_index('ZIPYEAR', inplace=True)

## double check the statistics again
print(refined_duration['duration_min'].describe(percentiles=[.01, .05, .25, .5, .75, .99]))

count    6.746080e+05
mean     3.579724e+04
std      2.045203e+05
min      1.666667e-02
1%       5.000000e+00
5%       2.176667e+01
25%      1.600000e+02
50%      2.808183e+03
75%      1.077531e+04
99%      8.265187e+05
max      6.872126e+06
Name: duration_min, dtype: float64


In [18]:
# filter for positive durations to avoid the rest of the 0s noise
duration_filtered = minimal311[minimal311['duration'] > 0]

# ok now group by ZIPYEAR and calculate multiple statistics at once
DurationFrame = duration_filtered.groupby('ZIPYEAR')['duration'].agg([
    ('avg_duration_min', 'mean'),
    ('p1_duration_min', lambda x: x.quantile(0.01)),
    ('p99_duration_min', lambda x: x.quantile(0.99))
])

# add the column for average duration in hours
DurationFrame['dur_avg_hrs'] = DurationFrame['avg_duration_min'] / 60

# add the column for average duration in days
DurationFrame['dur_avg_days'] = DurationFrame['dur_avg_hrs'] / 24

# check the results
print(DurationFrame.head())

            avg_duration_min  p1_duration_min  p99_duration_min  dur_avg_hrs  \
ZIPYEAR                                                                        
11212 2012      16425.334696        10.688667     252047.308000   273.755578   
11212 2013      15479.077594        18.596667     233440.075333   257.984627   
11212 2014      20005.854393        18.999167     260647.522500   333.430907   
11212 2015      17943.676919        18.380833     243920.602333   299.061282   
11212 2016      16392.337235        18.466167     188426.429667   273.205621   

            dur_avg_days  
ZIPYEAR                   
11212 2012     11.406482  
11212 2013     10.749359  
11212 2014     13.892954  
11212 2015     12.460887  
11212 2016     11.383568  


### 3c) district count data frame

In [19]:
print(len(minimal311['district'].unique()))
# 14 

# create the contingency table (cross-tabulation)
DistrictFrame = pd.crosstab(minimal311['ZIPYEAR'], minimal311['district'])


# check how it did
DistrictFrame.head()

14


district,3.0,11.0,33.0,34.0,35.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,45.0
ZIPYEAR,,,,,,,,,,,,,
11212 2012,0,1,0,0,0,1130,0,0,0,12050,5976,0,7
11212 2013,0,0,0,0,0,672,0,0,0,11350,6333,0,11
11212 2014,0,1,0,0,0,606,0,0,0,13758,6508,0,2
11212 2015,0,1,0,0,0,628,0,0,0,13336,7221,0,4
11212 2016,0,0,0,0,0,908,0,0,0,13848,7306,0,4


### 3d) precinct count data frame

In [20]:
print(len(minimal311['precinct'].unique()))
# 14 

# create the contingency table (cross-tabulation)
PrecinctFrame = pd.crosstab(minimal311['ZIPYEAR'], minimal311['precinct'])


# check how it did
PrecinctFrame.head()

12


precinct,0,67,69,71,72,73,76,77,78,83,90,104
ZIPYEAR,,,,,,,,,,,,
11212 2012,107,9422,3,8,0,9559,0,81,0,0,0,0
11212 2013,103,9141,6,10,0,9084,0,51,0,0,0,0
11212 2014,135,10625,3,10,0,10076,0,48,0,0,0,0
11212 2015,182,10399,3,2,0,10455,0,168,0,0,0,0
11212 2016,177,10000,4,2,0,11669,0,250,0,0,0,0


In [21]:
# check if all 33 rows have survived
print({len(AgencyFrame)})

# check the amt complaints is still right
print(f"total complaints in original: {len(minimal311)}")
print(f"total complaints in AgencyFrame: {AgencyFrame.values.sum()}")

# check for 'Unspecified' or '0' values that might be hiding
print(minimal311['precinct'].value_counts().head())

{33}
total complaints in original: 689770
total complaints in AgencyFrame: 689770
precinct
83    191771
78    164458
73    136729
67    108366
72     61019
Name: count, dtype: int64


### 3e) problem count data frame

In [22]:
# make a contingency table for the categorized problem groups
ProblemFrame = pd.crosstab(minimal311['ZIPYEAR'], minimal311['problem_group'])

# list columns
problem_cols = [
    'housing_building', 'noise_qol', 'sanitation_env', 'street_traffic', 
    'public_safety', 'parks_nature', 'finance_legal', 'business_consumer', 
    'tech_infrastructure', 'education', 'other_misc'
]

# reindex to ensure all columns exist and are in a consistent order
ProblemFrame = ProblemFrame.reindex(columns=problem_cols, fill_value=0)

# check the results
ProblemFrame.head()

problem_group,housing_building,noise_qol,sanitation_env,street_traffic,public_safety,parks_nature,finance_legal,business_consumer,tech_infrastructure,education,other_misc
ZIPYEAR,,,,,,,,,,,
11212 2012,10592,2702,1237,1895,464,346,433,274,1038,14,185
11212 2013,10021,2586,1104,2409,384,225,351,272,988,24,31
11212 2014,11522,2515,1358,2632,410,331,368,268,943,13,537
11212 2015,10783,2858,1329,3089,535,274,376,294,1099,8,564
11212 2016,10571,3346,1362,3477,780,264,324,241,1045,22,670



---

## 4) MERGING THE DATASETS 
### 4a) Prepare the columns which the datasets will be merged on

In [23]:
## MERGE PREP

# FIX: try to rename regardless of whether it has underscores or spaces
census = census.rename(columns={
    "zip_code_tabulation_area": "ZIP",
    "zip code tabulation area": "ZIP",
    "year": "YEAR"
})

# create ZIPYEAR
census['ZIPYEAR'] = census['ZIP'].astype(str) + ' ' + census['YEAR'].astype(str)

print("Columns now in census:", census.columns)
print("Unique years in census:", census['YEAR'].unique())



Columns now in census: Index(['total_pop', 'white_nonhisp', 'black', 'asian', 'hispanic',
       'median_hh_inc', 'per_capita_inc', 'gini_index', 'pop_below_poverty',
       'hh_on_snap', 'med_home_value', 'med_gross_rent', 'owner_occupied',
       'renter_occupied', 'vacant_units', 'bachelors_deg', 'masters_deg',
       'unemployed_count', 'foreign_born', 'moved_last_year', 'state', 'ZIP',
       'YEAR', 'ZIPYEAR'],
      dtype='str')
Unique years in census: [2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022]


### 4b) Create full 311 dataset
Merging the count versions we created above

In [24]:
# merge all the 311 count/summary frames together on their shared ZIPYEAR index
full311 = AgencyFrame \
    .join(DurationFrame, how='outer') \
    .join(ProblemFrame, how='outer')
   # .join(DistrictFrame, how='outer') changed our minds on this one
   # .join(PrecinctFrame, how='outer') ditto here

full311 = full311.reset_index()  # bring ZIPYEAR back as a column for merging

print(full311.shape)
full311.head()

# last thing before the merge- let's save just this data on its own
full311.to_csv('cleaned311.csv', index=False)

(33, 24)


### 4c) Merge the datasets!
The outcome here should be one dataframe with 33 rows (a row for each year, 2012-2022 (inclusive) for each of the three zip codes) and XX columns (however many 4b leaves us with). Everything should be merged on ZIPYEAR.

In [25]:
# print(full311['ZIPYEAR'])

# create ZIPYEAR in the census dataset (to match the 311 format)
census['ZIPYEAR'] = census['ZIP'].astype(str) + ' ' + census['YEAR'].astype(str)

# merge on ZIPYEAR
merged = pd.merge(census, full311, on='ZIPYEAR', how='inner')

# print(merged.shape)
# merged.head()

### 4d) final touches & save it

In [26]:


# move ZIPYEAR to the front
cols = ['ZIPYEAR'] + [col for col in merged.columns if col != 'ZIPYEAR']
merged = merged[cols]

# save to CSV
merged.to_csv('merged.csv', index=False)

print(merged.head())


      ZIPYEAR  total_pop  white_nonhisp    black   asian  hispanic  \
0  11212 2012    81267.0          749.0  66404.0  1107.0   12046.0   
1  11212 2013    84520.0          982.0  69739.0   818.0   11906.0   
2  11212 2014    87751.0         1018.0  71232.0   840.0   13539.0   
3  11212 2015    88668.0         1285.0  70612.0  1124.0   14619.0   
4  11212 2016    86469.0         1208.0  68375.0  1321.0   14664.0   

   median_hh_inc  per_capita_inc  gini_index  pop_below_poverty  ...  \
0        27901.0         15405.0      0.4965            27429.0  ...   
1        28348.0         16535.0      0.5001            28856.0  ...   
2        28146.0         15814.0      0.5028            30354.0  ...   
3        28207.0         15853.0      0.4975            30735.0  ...   
4        28495.0         17064.0      0.5173            29835.0  ...   

   noise_qol  sanitation_env  street_traffic  public_safety  parks_nature  \
0       2702            1237            1895            464          

In [27]:
## just for the presentation

print(merged.columns)

Index(['ZIPYEAR', 'total_pop', 'white_nonhisp', 'black', 'asian', 'hispanic',
       'median_hh_inc', 'per_capita_inc', 'gini_index', 'pop_below_poverty',
       'hh_on_snap', 'med_home_value', 'med_gross_rent', 'owner_occupied',
       'renter_occupied', 'vacant_units', 'bachelors_deg', 'masters_deg',
       'unemployed_count', 'foreign_born', 'moved_last_year', 'state', 'ZIP',
       'YEAR', 'biz_tech', 'doe', 'fin_prop', 'infr_env', 'safety', 'school',
       'ss_hous', 'avg_duration_min', 'p1_duration_min', 'p99_duration_min',
       'dur_avg_hrs', 'dur_avg_days', 'housing_building', 'noise_qol',
       'sanitation_env', 'street_traffic', 'public_safety', 'parks_nature',
       'finance_legal', 'business_consumer', 'tech_infrastructure',
       'education', 'other_misc'],
      dtype='str')
